# **Documents Summarizer**

- Takes in a PDF or word document
- Summarizes the data into bullet points
- Highlights key items
- Add a portion to create charts if numbers discussed.

ToDO:
- Make scanned pdfs readable
- Add logic to handle Docs
- Add logic to chunk docs over certain size
- Add logic to display graphs

In [1]:
# imports

import os
from pypdf import PdfReader
from dotenv import load_dotenv
# import docx2txt
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
openai = OpenAI()

# **Prompts**

In [4]:
system_prompt = """
You are a business analyst with legal experience responsible for reviewing and providing summaries of documents given to you.

You will be given one or more documents, in an array of strings format where each item is a document, covering a variety of areas, but expected to be focused on financial or legal data.

If multiple documents are given, assume they are related and provide a summary of the whole.
If it is clear documents are not related, then provide a summary of each document separately and highlight they are not related at the start of your response.

If you are given financial data or data covering a trend, generate an  appropriate chart to help visualize the data.

Format the response to begin with a brief summary overview then outline relevant sections into a bullet point format for readability.

"""

user_prompt = """
    Here are a few documents. Provide a summary of the contents.
"""

In [5]:
def messages_for_ai(docText):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + f"{docText}"}
    ]

In [10]:
def docsToPrompts(*docPaths):

    docText = []

    for _, docPath in enumerate(docPaths):
        if docPath.endswith(".pdf"):
            reader=PdfReader(docPath)
            result= "\n".join([p.extract_text() for p in reader.pages])
            docText.append(result)
        # else if docPath.endswith*".docx"
        else:
            with open(docPath, encoding="utf-8") as f:
                docText.append(f.read())
    return docText

    


In [7]:
def dataSummarizer(*docPaths):
    doc_response_collection = docsToPrompts(*docPaths)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for_ai(doc_response_collection)
    )

    return response.choices[0].message.content

In [8]:
document_1 = "/Users/cgoose/Documents/Coding/llm_engineering/week1/carl-contribution/The_Problem_With_Grading.txt"

document_2 = "/Users/cgoose/Documents/Coding/llm_engineering/week1/carl-contribution/Kelly_Vickie.pdf"

In [11]:
docsToPrompts(document_1)

['My son’s binder was a mess. Loose papers were falling out, others looked like they had been balled up or stepped on, some more than once. The binder itself was bent in one corner. But he was a seventh-grader and to him, it looked just fine.\n\nUnfortunately, his seventh-grade math teacher didn’t agree and deducted points from his grade for being messy. This same teacher also took off points when homework was completed with something other than a pencil or if a student needed a second copy of an assignment. If a student was asked to move their seat during class, she slashed five points. Points were earned back if a parent signed the list of rules, and it was returned in a timely manner.\n\nBeing organized and not misbehaving in class are skills students need to figure out, for sure, and I certainly wanted my son to be neater, but factoring these behaviors into grades — especially for middle-schoolers just learning to come into their own — didn’t make sense to me.\n\nAnd so, when I lea

In [12]:
def display_summary_docs(*docsToPrompts):
    summary = dataSummarizer(*docsToPrompts)
    display(Markdown(summary))

In [13]:
display_summary_docs(document_1, document_2)

The provided documents include two distinct but education-related analyses:

1. **An in-depth article discussing "Rethinking Grading" practices in K-12 education, focusing on the challenges, inequities, and potential reforms to traditional grading systems.**  
2. **A comprehensive doctoral dissertation titled "A Quantitative Study of Course Grades and Retention Comparing Online and Face-to-Face Classes," examining higher education outcomes between online and traditional course delivery modes in a bachelor's degree completion program.**

---

### Overall Summary Overview

- The first document is a narrative and research-informed discussion about the inadequacies and inconsistencies of traditional grading systems in middle and high school settings. It explores how grades can misrepresent student mastery, incorporate non-academic factors, engender stress and inequity, and suggests alternatives such as competency-based grading and revision opportunities.

- The second document is a rigorous quantitative study analyzing whether course grades, course retention, and program retention differ significantly between online and face-to-face course delivery among students in a Technology Administration bachelor's degree completion program at Washburn University. The study finds **no statistically significant difference** between online and traditional students in grades or retention, supporting online education as an effective delivery method.

---

### Detailed Summaries

#### 1. **Rethinking Grading - Challenges and Alternatives in K-12**

- **Context & Problems:**  
  - Traditional grading often includes non-academic factors (e.g., neatness, behavior) affecting students' grades, which may not accurately reflect mastery or learning.  
  - Grades lack consistency across teachers and subjects, causing "grade fog" where a letter grade (A, B, C, etc.) does not clearly communicate mastery.  
  - The use of averaging grades and zero scores can disproportionately penalize students, affecting motivation and performance.  
  - Including behavior or participation in grades can mask gaps in content mastery, potentially setting students up for failure later.  
  - The stress of grades and GPA focus may lead to unhealthy perfectionism, anxiety, or disengagement from learning.

- **Current Impacts:**  
  - Students face high cognitive load trying to navigate inconsistent grading policies.  
  - Parents and counselors misinterpret grades, confusing behavior or effort with academic success.  
  - Some students may get a false sense of proficiency due to inflated grades from non-academic points.

- **Reform Efforts & Alternatives:**  
  - Schools like Greater Dayton and Revere High are experimenting with competency-based models using simple binary or scaled grading (mastered/in progress or 1-4 scale).  
  - Policies to allow revisions and retakes focus grading on content mastery rather than compliance or punctuality.  
  - Habits-of-work scales separate behavioral factors from academic achievement, providing clearer feedback.  
  - Rethinking homework policies — treating homework as practice rather than graded summative assessment.  
  - Emphasis on teacher professional development and collaborative policy setting to ensure consistency and equity.

- **Broader Significance:**  
  - "Grading for Equity" is gaining traction to improve student achievement, reduce disparities, and support teacher retention by reducing the policing role of grading.  
  - The movement aligns grading more with assessment and learning feedback rather than punitive measures or sorting mechanisms.

---

#### 2. **Doctoral Research Study: Comparing Online and Face-to-Face Education Outcomes**

- **Background:**  
  - Online education has grown rapidly as an alternative to traditional classroom instruction, especially in degree completion programs for adult learners with associate degrees.  
  - Prior research generally finds no significant difference in learning outcomes (grades) between delivery methods, but retention rates for online students have been questioned.

- **Research Purpose & Questions:**  
  - To determine if statistically significant differences exist in course grades, course retention, and program retention between online and face-to-face students in the Technology Administration (TA) bachelor's completion program at Washburn University.

- **Methodology:**  
  - Quantitative design using archival data from 2002-2008.  
  - Sample: 885 students in TA 300 and TA 310 courses taught online and face-to-face by two instructors.  
  - Analysis: Two-factor ANOVA for grades by delivery mode and instructor; Chi-square tests for retention comparisons.  
  - Program retention tracked based on degree completion or continued enrollment.

- **Key Findings:**  
  - **Grades:** No significant difference in grades between online and face-to-face students. No effect by instructor or interaction effects.  
  - **Course Retention:** No significant difference between online and face-to-face student retention in courses (~91-94%).  
  - **Program Retention:** No significant difference between students starting online vs. face-to-face (~92% retention).  
  - Descriptive statistics showed the sample included a mix of ages (mean ~31 years), with more females proportionally choosing online courses.

- **Implications:**  
  - Online delivery is as effective as face-to-face in terms of student achievement and retention for this program.  
  - Supports further expansion of online courses and degree programs.  
  - Recommends further research on different populations, disciplines, and factors influencing retention.

---

### Visualizing Data from the Quantitative Study

Below is a summary chart visualizing the comparison of course retention and program retention rates between online and face-to-face students based on the study data.

| Metric             | Course Retention % | Program Retention % |
|--------------------|--------------------|---------------------|
| **Online Students** | 93.92%             | 91.57%              |
| **Face-to-Face**    | 90.89%             | 92.96%              |

*(No statistically significant difference detected, p > 0.05)*

---

### Final Notes

- These two documents complement each other by addressing educational quality and equity from K-12 grading practices to higher education delivery models.  
- While the first argues for reform in evaluation and reporting to better represent student mastery and reduce inequities, the second supports validity and equivalence of online education outcomes in higher education. Both highlight the importance of transparent, fair, and meaningful measures of student learning.